# YouTube Playlist Tracker Notebook
Use this notebook for a run-once workflow to validate API access, fetch playlist stats, and optionally save one snapshot.

## Before You Run
1. Open this folder in VS Code: `C:\repos\youtube-playlist-tracker`
2. Open Terminal in VS Code (`Terminal -> New Terminal`)
3. Install packages once: `pip install -r requirements.txt`
4. Open this notebook and run cells in order
5. If prompted for kernel, choose your Python environment

## Run Order (Simple)
1. Run Cell 2 to install/import dependencies.
2. Run Cell 3 to configure inputs and API key handling.
3. Run Cell 4 to discover playlists (optional if you already have a playlist URL/ID).
4. Run Cell 5 to resolve the playlist ID.
5. Run Cell 6 to fetch video statistics (main validation step).
6. Run Cell 7 only if you want to save a snapshot.
7. Run Cell 8 to view snapshot history summary.

## API Key (Simple)
- Option A: set environment variable `YOUTUBE_API_KEY` before running the notebook.
- Option B: leave env var unset and Cell 3 will prompt for key input (hidden).

## Streamlit App Command
Run Streamlit from the VS Code terminal, not from notebook cells:
`streamlit run app.py`

Tip: keep `SAVE_SNAPSHOT = False` for validation-only runs.

In [9]:
# Optional install (uncomment if needed)
%pip install -q -r requirements.txt

import sys
from pathlib import Path

import pandas as pd

from youtube_tracker.storage import append_snapshot, load_snapshot_history
from youtube_tracker.youtube_api import (
    YouTubeApiError,
    discover_channel_playlists,
    extract_playlist_id,
    fetch_playlist_videos_with_stats,
)

print('Python:', sys.version.split()[0])
print('pandas:', pd.__version__)

Note: you may need to restart the kernel to use updated packages.
Python: 3.12.4
pandas: 2.2.2


In [13]:
# Configuration (safe key handling)
import os
from getpass import getpass

API_KEY = os.getenv("YOUTUBE_API_KEY", "").strip()
if not API_KEY:
    API_KEY = getpass("Enter YouTube API key (input hidden): ").strip()

CHANNEL_INPUT = "https://www.youtube.com/@tedxnormal"  # Example: https://www.youtube.com/@yourhandle or UC...
PLAYLIST_INPUT = "https://www.youtube.com/watch?v=7nAU7KXXHEI&list=PL3ONZXacNoK57cL1En521u37CYrMPvuXY"  # Example: https://www.youtube.com/playlist?list=PL...
SAVE_SNAPSHOT = False

if not API_KEY:
    raise ValueError("API key is required. Set YOUTUBE_API_KEY or enter it when prompted.")

repo_root = Path.cwd()
print("Repo root:", repo_root)

Repo root: c:\repos\youtube-playlist-tracker


In [14]:
# Discover public playlists for a channel
playlists_df = pd.DataFrame()

if CHANNEL_INPUT.strip():
    try:
        playlists = discover_channel_playlists(API_KEY, CHANNEL_INPUT)
        playlists_df = pd.DataFrame(playlists)
        print(f'Discovered {len(playlists_df)} playlists')
        display(playlists_df.head(20))
    except YouTubeApiError as exc:
        raise RuntimeError(f'Playlist discovery failed: {exc}') from exc
else:
    print('CHANNEL_INPUT is blank; skipping discovery.')

Discovered 12 playlists


,playlist_id,title,video_count
0,PL3ONZXacNoK57cL1En521u37CYrMPvuXY,ALL TEDxNormal Videos,96
1,PL3ONZXacNoK7-r0TWwCWODwM9eURNlkdJ,TEDxNormal 2015-2025: TED Editor's Picks,2
2,PL3ONZXacNoK4umU2hpmnH3OjaQrJgmJpF,TEDxNormal 2015-2025: Top Ten Most Viewed,10
3,PL3ONZXacNoK6gZsVdsvZ0fM7-MQTkNAmP,TEDxNormal 2015: Anything But Normal,18
4,PL3ONZXacNoK7CFDLdwtQ6h60rVfp-YysR,TEDxNormal 2016: Anything But Normal,17
5,PL3ONZXacNoK5gWH9X3eDhId_RLqDZ2TXY,TEDxNormal 2017: Anything But Normal,9
6,PL3ONZXacNoK5QUT-XkQFiN2EI4x02tDLs,TEDxNormal 2018: Engage,9
7,PL3ONZXacNoK5GCPvoI3maFKVM7Bu2DG9h,TEDxNormal 2019: Authentic,8
8,PL3ONZXacNoK5hq20PDGUysXUCDGVlXZLg,TEDxNormal 2021: Unintended Knowledge,8
9,PL3ONZXacNoK5q0EAg-VMR4R0tJYSCHbzt,TEDxNormal 2022: The Power of Two,10


In [15]:
# Choose playlist to fetch
selected_playlist_id = ""

if PLAYLIST_INPUT.strip():
    selected_playlist_id = extract_playlist_id(PLAYLIST_INPUT.strip())
    if not selected_playlist_id:
        raise ValueError('PLAYLIST_INPUT is not a valid playlist URL or ID.')
elif not playlists_df.empty:
    selected_playlist_id = playlists_df.iloc[0]['playlist_id']
    print('PLAYLIST_INPUT not set. Using first discovered playlist:', selected_playlist_id)
else:
    raise ValueError('Set PLAYLIST_INPUT or provide CHANNEL_INPUT that returns playlists.')

selected_playlist_id

'PL3ONZXacNoK57cL1En521u37CYrMPvuXY'

In [16]:
# Fetch video statistics for selected playlist
try:
    stats_df = fetch_playlist_videos_with_stats(API_KEY, selected_playlist_id)
except YouTubeApiError as exc:
    raise RuntimeError(f'Video stats fetch failed: {exc}') from exc

print(f'Rows: {len(stats_df)}')
display(stats_df.head(20))

Rows: 96


,video_id,title,channel_title,published_at,view_count,like_count,comment_count
0,Fom14XGMFHA,How to change your limiting beliefs for more s...,TEDx Talks,2015-12-08T17:35:36Z,656536,9703,359
1,Qm172DbaSbc,How Simplification is the Key to Change | Lisa...,TEDx Talks,2015-12-07T20:54:26Z,415648,4108,89
2,bCteZqlwf-k,Check Yourself - Accountability | Charlie Joh...,TEDx Talks,2018-01-09T20:10:23Z,326389,5297,151
3,jVAVWhjxL48,Don’t Be Afraid of Messaging Extraterrestrial ...,TEDx Talks,2020-02-03T17:38:33Z,276547,4044,1010
4,ybVrejffXpg,Feeling Better and Getting Better with Photos ...,TEDx Talks,2023-03-08T17:41:14Z,129362,11888,445
5,9tNsNkzEzbw,"How to Be a Man, A Woman's Guide | Elizabeth P...",TEDx Talks,2016-11-15T19:58:36Z,87291,1116,523
6,WimcBDghNhw,Reframing Reproductive Rights: Going Beyond Pr...,TEDx Talks,2019-01-07T20:53:58Z,83485,961,580
7,sA0uwuCA9d0,The Way You Are Coping With Burnout is Keeping...,TEDx Talks,2025-06-13T15:11:56Z,74712,5635,71
8,nma0Gey-KEU,Mental Illness is an Asset | Mike Veny | TEDxN...,TEDx Talks,2015-12-07T18:21:51Z,58436,421,26
9,xR7Z_3aE5cE,Why aren't more of us engaged at work? | Jeff ...,TEDx Talks,2015-12-07T19:46:35Z,55826,385,9


In [17]:
# Optionally save snapshot to local history
snapshot_time = None
if SAVE_SNAPSHOT and not stats_df.empty:
    playlist_title = ""
    if not playlists_df.empty and selected_playlist_id in set(playlists_df['playlist_id']):
        playlist_title = playlists_df.loc[playlists_df['playlist_id'] == selected_playlist_id, 'title'].iloc[0]
    snapshot_time = append_snapshot(
        playlist_id=selected_playlist_id,
        playlist_title=playlist_title,
        df=stats_df,
    )
    print('Snapshot saved at:', snapshot_time)
else:
    print('Snapshot not saved (SAVE_SNAPSHOT is False or no rows).')

Snapshot not saved (SAVE_SNAPSHOT is False or no rows).


In [18]:
# Historical summary for selected playlist
history_df = load_snapshot_history()

if history_df.empty:
    print('No snapshot history found yet.')
else:
    selected_history = history_df[history_df['playlist_id'] == selected_playlist_id].copy()
    if selected_history.empty:
        print('No snapshots found for selected playlist yet.')
    else:
        summary_df = (
            selected_history.groupby('snapshot_time', as_index=False)
            .agg(
                video_rows=('video_id', 'count'),
                total_views=('view_count', 'sum'),
                total_likes=('like_count', 'sum'),
                total_comments=('comment_count', 'sum'),
            )
            .sort_values('snapshot_time', ascending=False)
        )
        display(summary_df.head(30))

,snapshot_time,video_rows,total_views,total_likes,total_comments
0,2026-05-30T04:29:28.847782+00:00,96,2654904,52210,3976


## Optional: Launch Streamlit App (From Terminal)
If you also want the web dashboard, use VS Code terminal:

1. Open terminal: `Terminal -> New Terminal`
2. Run: `streamlit run app.py`
3. Open the localhost URL shown in terminal (usually `http://localhost:8501`)

This notebook is for run-once validation and snapshot checks.